In [5]:
from pathlib import Path
import re
import csv
from datetime import datetime
from collections import Counter

# --- Config ---
PROJECT_ROOT = Path(r"D:\SIT374\WalkBuddy-T2-2026\ML_side")
INTERIM_MAPPED_DIR = PROJECT_ROOT / "datasets" / "interim_mapped"
REPORTS_DIR = PROJECT_ROOT / "datasets" / "reports"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert INTERIM_MAPPED_DIR.exists(), f"Path not found: {INTERIM_MAPPED_DIR}"
assert REPORTS_DIR.exists(), f"Path not found: {REPORTS_DIR}"
print(f"Reading from: {INTERIM_MAPPED_DIR}")
print(f"Reports go to: {REPORTS_DIR}")

Reading from: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim_mapped
Reports go to: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports


In [6]:
def scan_naming_conventions(root: Path):
    all_files = [f for f in root.rglob("*") if f.is_file()]
    images = [f for f in all_files if f.suffix.lower() in IMAGE_EXTS]
    labels = [f for f in all_files if f.suffix.lower() == ".txt"]

    image_stems = {f.stem: f for f in images}
    label_stems = {f.stem: f for f in labels}

    ext_counter = Counter(f.suffix for f in images)
    images_without_labels = sorted(set(image_stems) - set(label_stems))
    labels_without_images = sorted(set(label_stems) - set(image_stems))

    illegal_char_pattern = re.compile(r"[^A-Za-z0-9_\-]")
    illegal_examples = [f.name for f in images if illegal_char_pattern.search(f.stem)]

    delimiter_counter = Counter()
    for f in images:
        has_underscore = "_" in f.stem
        has_hyphen = "-" in f.stem
        if has_underscore and has_hyphen:
            delimiter_counter["mixed"] += 1
        elif has_underscore:
            delimiter_counter["underscore_only"] += 1
        elif has_hyphen:
            delimiter_counter["hyphen_only"] += 1
        else:
            delimiter_counter["no_delimiter"] += 1

    prefix_counter = Counter()
    for f in images:
        first_token = re.split(r"[_\-]", f.stem)[0]
        prefix_counter[first_token] += 1

    return {
        "total_images": len(images),
        "total_labels": len(labels),
        "ext_counter": ext_counter,
        "images_without_labels": images_without_labels,
        "labels_without_images": labels_without_images,
        "illegal_examples": illegal_examples,
        "delimiter_counter": delimiter_counter,
        "prefix_counter": prefix_counter,
    }

results = scan_naming_conventions(INTERIM_MAPPED_DIR)
print("Scan complete — no files modified.")

Scan complete — no files modified.


In [7]:
print(f"Total images: {results['total_images']}")
print(f"Total labels: {results['total_labels']}")
print()
print("Extension usage:")
for ext, count in results["ext_counter"].most_common():
    print(f"  {ext}: {count}")
print()
print(f"Images without matching label: {len(results['images_without_labels'])}")
print(f"Labels without matching image: {len(results['labels_without_images'])}")
print()
print(f"Filenames with unusual characters: {len(results['illegal_examples'])}")
print()
print("Delimiter style distribution:")
for style, count in results["delimiter_counter"].most_common():
    print(f"  {style}: {count}")
print()
print(f"Distinct filename prefixes: {len(results['prefix_counter'])}")
print("Top 15 by frequency:")
for prefix, count in results["prefix_counter"].most_common(15):
    print(f"  {prefix}: {count}")

Total images: 35633
Total labels: 35633

Extension usage:
  .jpg: 35475
  .png: 121
  .jpeg: 30
  .JPG: 7

Images without matching label: 0
Labels without matching image: 0

Filenames with unusual characters: 20535

Delimiter style distribution:
  underscore_only: 22727
  mixed: 11349
  no_delimiter: 1311
  hyphen_only: 246

Distinct filename prefixes: 3752
Top 15 by frequency:
  IMG: 12054
  y2mate: 1616
  img: 1537
  outdoor: 1367
  fourway: 1281
  20240909: 1108
  image: 1042
  Screenshot: 831
  20240907: 755
  night: 565
  20240914: 556
  2008: 436
  2009: 436
  autobus: 399
  crosswalk: 378


In [8]:
def normalize_ext(suffix: str) -> str:
    s = suffix.lower()
    if s == ".jpeg":
        s = ".jpg"
    return s

image_files = sorted(
    [f for f in INTERIM_MAPPED_DIR.rglob("*") if f.is_file() and f.suffix.lower() in IMAGE_EXTS],
    key=lambda f: str(f.relative_to(INTERIM_MAPPED_DIR))
)
label_lookup = {f.stem: f for f in INTERIM_MAPPED_DIR.rglob("*.txt")}

mapping = []
for idx, img in enumerate(image_files, start=1):
    label = label_lookup.get(img.stem)
    if label is None:
        continue

    new_stem = f"wb_{idx:06d}"
    mapping.append({
        "old_image_path": str(img.relative_to(INTERIM_MAPPED_DIR)),
        "old_label_path": str(label.relative_to(INTERIM_MAPPED_DIR)),
        "new_image_name": new_stem + normalize_ext(img.suffix),
        "new_label_name": new_stem + ".txt",
    })

print(f"Mapping built for {len(mapping)} pairs.")
for row in mapping[:5]:
    print(" ", row)

Mapping built for 35633 pairs.
  {'old_image_path': 'kaggle\\indoor_object_detection\\test\\images\\1003.png', 'old_label_path': 'kaggle\\indoor_object_detection\\test\\images\\1003.txt', 'new_image_name': 'wb_000001.png', 'new_label_name': 'wb_000001.txt'}
  {'old_image_path': 'kaggle\\indoor_object_detection\\test\\images\\1014.png', 'old_label_path': 'kaggle\\indoor_object_detection\\test\\images\\1014.txt', 'new_image_name': 'wb_000002.png', 'new_label_name': 'wb_000002.txt'}
  {'old_image_path': 'kaggle\\indoor_object_detection\\test\\images\\1015.png', 'old_label_path': 'kaggle\\indoor_object_detection\\test\\images\\1015.txt', 'new_image_name': 'wb_000003.png', 'new_label_name': 'wb_000003.txt'}
  {'old_image_path': 'kaggle\\indoor_object_detection\\test\\images\\1020.png', 'old_label_path': 'kaggle\\indoor_object_detection\\test\\images\\1020.txt', 'new_image_name': 'wb_000004.png', 'new_label_name': 'wb_000004.txt'}
  {'old_image_path': 'kaggle\\indoor_object_detection\\test\\

In [9]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
mapping_csv_path = REPORTS_DIR / f"filename_mapping_{timestamp}.csv"

with open(mapping_csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["old_image_path", "old_label_path", "new_image_name", "new_label_name"])
    writer.writeheader()
    writer.writerows(mapping)

print(f"Mapping saved to: {mapping_csv_path}")

Mapping saved to: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports\filename_mapping_20260809_055116.csv


In [10]:
EXECUTE_RENAME = True

if not EXECUTE_RENAME:
    raise RuntimeError("EXECUTE_RENAME is False — review the mapping CSV first.")

renamed = 0
for row in mapping:
    old_img = INTERIM_MAPPED_DIR / row["old_image_path"]
    old_label = INTERIM_MAPPED_DIR / row["old_label_path"]
    old_img.rename(old_img.parent / row["new_image_name"])
    old_label.rename(old_label.parent / row["new_label_name"])
    renamed += 1

print(f"Renamed {renamed} pairs. Old names recorded in: {mapping_csv_path}")

Renamed 35633 pairs. Old names recorded in: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports\filename_mapping_20260809_055116.csv


In [12]:
missing_images = []
missing_labels = []
unexpected_leftovers = []

for row in mapping:
    new_img = INTERIM_MAPPED_DIR / Path(row["old_image_path"]).parent / row["new_image_name"]
    new_label = INTERIM_MAPPED_DIR / Path(row["old_label_path"]).parent / row["new_label_name"]
    old_img = INTERIM_MAPPED_DIR / row["old_image_path"]
    old_label = INTERIM_MAPPED_DIR / row["old_label_path"]

    if not new_img.exists():
        missing_images.append(row["new_image_name"])
    if not new_label.exists():
        missing_labels.append(row["new_label_name"])
    if old_img.exists():
        unexpected_leftovers.append(str(old_img))
    if old_label.exists():
        unexpected_leftovers.append(str(old_label))

print(f"Checked {len(mapping)} pairs")
print(f"Missing new images: {len(missing_images)}")
print(f"Missing new labels: {len(missing_labels)}")
print(f"Old files still present (should be 0): {len(unexpected_leftovers)}")

all_new_images = [f for f in INTERIM_MAPPED_DIR.rglob("*") if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
all_new_labels = list(INTERIM_MAPPED_DIR.rglob("*.txt"))
print(f"Total images now: {len(all_new_images)}")
print(f"Total labels now: {len(all_new_labels)}")

Checked 35633 pairs
Missing new images: 0
Missing new labels: 0
Old files still present (should be 0): 0
Total images now: 35633
Total labels now: 35633


In [14]:
def summarize_interim_mapped(root: Path):
    all_files = [f for f in root.rglob("*") if f.is_file()]
    images = [f for f in all_files if f.suffix.lower() in IMAGE_EXTS]
    labels = [f for f in all_files if f.suffix.lower() == ".txt"]

    ext_counter = Counter(f.suffix.lower() for f in images)

    # naming convention compliance check
    wb_pattern = re.compile(r"^wb_\d{6}$")
    non_compliant = [f.name for f in images if not wb_pattern.match(f.stem)]

    # per-class instance count (class_id = first field of each YOLO line)
    class_counter = Counter()
    unreadable_labels = []
    for lbl in labels:
        try:
            with open(lbl, "r") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    class_id = line.split()[0]
                    class_counter[class_id] += 1
        except Exception:
            unreadable_labels.append(lbl.name)

    return {
        "total_images": len(images),
        "total_labels": len(labels),
        "ext_counter": ext_counter,
        "non_compliant_names": non_compliant,
        "class_counter": class_counter,
        "unreadable_labels": unreadable_labels,
    }


summary = summarize_interim_mapped(INTERIM_MAPPED_DIR)

print("=== interim_mapped/ Summary (post-standardization) ===")
print(f"Total images: {summary['total_images']}")
print(f"Total labels: {summary['total_labels']}")
print()
print("Extension distribution:")
for ext, count in summary["ext_counter"].most_common():
    print(f"  {ext}: {count}")
print()
print(f"Filenames NOT matching wb_NNNNNN convention: {len(summary['non_compliant_names'])}")
if summary["non_compliant_names"]:
    print("  sample:", summary["non_compliant_names"][:5])
print()
print("Per-class instance counts:")
for class_id, count in sorted(summary["class_counter"].items(), key=lambda x: -x[1]):
    print(f"  class {class_id}: {count}")
print()
print(f"Unreadable label files: {len(summary['unreadable_labels'])}")

# --- Write summary to reports/ ---
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_path = REPORTS_DIR / f"standardization_summary_{timestamp}.csv"

with open(summary_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerow(["total_images", summary["total_images"]])
    writer.writerow(["total_labels", summary["total_labels"]])
    for ext, count in summary["ext_counter"].most_common():
        writer.writerow([f"ext_{ext}", count])
    writer.writerow(["non_compliant_filenames", len(summary["non_compliant_names"])])
    writer.writerow(["unreadable_labels", len(summary["unreadable_labels"])])
    for class_id, count in sorted(summary["class_counter"].items(), key=lambda x: -x[1]):
        writer.writerow([f"class_{class_id}_count", count])

print(f"\nSummary saved to: {summary_path}")

=== interim_mapped/ Summary (post-standardization) ===
Total images: 35633
Total labels: 35633

Extension distribution:
  .jpg: 35512
  .png: 121

Filenames NOT matching wb_NNNNNN convention: 0

Per-class instance counts:
  class 0: 25714
  class 2: 21839
  class 7: 16407
  class 4: 8608
  class 5: 5665
  class 6: 5307
  class 3: 5072
  class 1: 3414

Unreadable label files: 0

Summary saved to: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports\standardization_summary_20260809_061028.csv


---

End of standardization

---